# AI College Academic Assistant — RAG Setup & Testing

This notebook covers the **retrieval half** of the project, end to end, in Google Colab:

1. Install packages
2. Load the college PDFs
3. Extract text with page metadata
4. Split into chunks
5. Generate embeddings
6. Build and search a FAISS index
7. Inspect retrieved chunks
8. Test sample college questions
9. Compare retrieval against a no-context baseline

## Where Ollama runs

Ollama does **not** run natively in Colab — it is a local desktop/server application, and a
Colab runtime cannot reach the Ollama server on your laptop over the network by default.

The split is deliberate:

| Component | Where it runs |
|---|---|
| PDF loading, extraction, chunking | Colab and local (same code) |
| Embeddings + FAISS retrieval | Colab and local (same code) |
| Ollama LLM generation | **Local machine only** |
| LangGraph workflow + Streamlit UI | **Local machine only** |

So this notebook deliberately stops at retrieval. Cell 9 shows what the retrieved context
looks like, which is what the local app feeds to Ollama. The LLM itself is exercised locally
by the pytest suite (`tests/test_end_to_end.py`) and the Streamlit app.

## Using the real college PDFs

Upload your PDFs into a `data/` folder in Colab (Folder icon in the sidebar → upload).
Set `DATA_DIR = "/content/data"` below. Everything else is automatic.

If `data/` is empty, cell 6 generates dummy documents so the pipeline can still be
tested. The dummy files are clearly labelled and are **not** real college information.

To reproduce the local knowledge base exactly, drag your local `data/` folder in — the
same FAISS-building code runs here as on your machine.

## 1. Install packages

In [ ]:
!pip install -q langchain langchain-community langchain-text-splitters langchain-huggingface pypdf faiss-cpu sentence-transformers reportlab matplotlib pandas matplotlib

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os, sys, glob, shutil, zipfile
import pandas as pd

print("Python", sys.version.split()[0])
print("pandas", pd.__version__)

## 2. Add the college PDFs

Upload the real college PDFs using the folder button in Colab, then point `DATA_DIR` at them.

If you have no PDFs to hand, run the next cell to generate the same placeholder documents the
local project uses.

In [ ]:
DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

def list_pdfs(path):
    files = sorted(glob.glob(os.path.join(path, "**", "*.pdf"), recursive=True))
    for f in files:
        print(f"  {os.path.basename(f):50s} {os.path.getsize(f)/1024:8.1f} KB")
    print(f"\n{len(files)} PDF(s) found")
    return files

pdf_files = list_pdfs(DATA_DIR)

In [ ]:
# Optional: generate dummy documents if data/ is empty, so the pipeline can be tested.
# These are NOT real college information - replace them with your own PDFs.
if not pdf_files:
    print("No PDFs found — generating dummy documents for testing.\n")
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet
    from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer

    banner = ("DUMMY DATA - TESTING ONLY. NOT real college information. "
              "Replace this file with your actual college document.")
    docs = {
        "Academic_Regulations_DUMMY.pdf": [
            ("Attendance", "Students should maintain at least 75% attendance in each course to be eligible for semester-end examinations."),
            ("Credits", "Students must complete the required credits specified by their programme curriculum."),
        ],
        "Examination_Guidelines_DUMMY.pdf": [
            ("Exam Eligibility", "Students must satisfy attendance and other academic requirements before appearing for the semester-end examination."),
            ("Registration", "Eligible students must complete examination registration within the announced deadline."),
        ],
        "Internship_Guidelines_DUMMY.pdf": [
            ("Eligibility", "Students must satisfy programme-specific eligibility requirements before undertaking an internship."),
            ("Documents", "Typical documentation may include an offer letter, joining confirmation, completion certificate, work report and student report."),
        ],
        "Academic_Calendar_DUMMY.pdf": [
            ("Internal Assessment", "Dummy Internal Assessment 1: 15 September. Dummy Internal Assessment 2: 20 October."),
            ("Semester Start", "Dummy semester starts on 01 August."),
        ],
        "Syllabus_DUMMY.pdf": [
            ("Database Management Systems", "Unit 1: Database concepts. Unit 2: Relational model and SQL. Unit 3: Normalization. Unit 4: Transactions. Unit 5: Indexing."),
        ],
        "Student_FAQ_DUMMY.pdf": [
            ("Q: What attendance is required?", "A: The dummy academic regulations state a minimum of 75% attendance for examination eligibility."),
        ],
    }
    styles = getSampleStyleSheet()
    for name, sections in docs.items():
        title = name.replace("_DUMMY.pdf", "").replace("_", " ")
        story = [Paragraph(title, styles["Title"]),
                 Paragraph(f"<i>{banner}</i>", styles["Normal"]), Spacer(1, 16)]
        for h, b in sections:
            story += [Paragraph(h, styles["Heading2"]), Paragraph(b, styles["Normal"]), Spacer(1, 12)]
        SimpleDocTemplate(os.path.join(DATA_DIR, name), pagesize=A4).build(story)

    pdf_files = list_pdfs(DATA_DIR)

## 3. Extract text with page metadata

One record per page, keeping the source filename and page number so every chunk can be cited.
Pages with almost no text are flagged — they are usually scanned images.

In [ ]:
from pypdf import PdfReader

def load_pdfs(data_dir, min_chars_per_page=50):
    pages, sparse = [], []
    for path in sorted(glob.glob(os.path.join(data_dir, "**", "*.pdf"), recursive=True)):
        name = os.path.basename(path)
        try:
            reader = PdfReader(path)
        except Exception as exc:
            print(f"  ! {name}: unreadable ({exc})")
            continue
        for i, page in enumerate(reader.pages):
            text = (page.extract_text() or "").strip()
            if len(text) < min_chars_per_page:
                sparse.append((name, i + 1))
            if text:
                pages.append({"page_content": text, "source": name, "page": i + 1})
        print(f"  + {name}: {len(reader.pages)} page(s)")
    if sparse:
        print(f"\n  ! Sparse pages (likely scans): {sparse}")
    return pages

pages = load_pdfs(DATA_DIR)
print(f"\n{len(pages)} page(s) of text extracted")

pd.DataFrame([{"source": p["source"], "page": p["page"],
               "chars": len(p["page_content"])} for p in pages])

In [ ]:
print(pages[0]["page_content"][:600])

## 4. Split into chunks

Overlapping chunks so a sentence spanning a boundary stays retrievable from both sides.
Metadata rides along with every chunk.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

CHUNK_SIZE, CHUNK_OVERLAP = 800, 120

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

docs = []
for page in pages:
    for chunk in splitter.split_text(page["page_content"]):
        chunk = chunk.strip()
        if len(chunk) >= 20:
            docs.append(Document(page_content=chunk,
                                 metadata={"source": page["source"], "page": page["page"]}))

print(f"{len(pages)} pages -> {len(docs)} chunks")
print(f"mean chunk size: {sum(len(d.page_content) for d in docs)/len(docs):.0f} chars")
docs[0]

## 5. Generate embeddings

`all-MiniLM-L6-v2` — small, fast, and CPU-friendly. The first run downloads ~90 MB.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)

probe = embeddings.embed_query("attendance requirement")
print(f"dimensions: {len(probe)}")
print(f"magnitude:   {sum(x**2 for x in probe)**0.5:.4f}  (should be ~1.0)")

## 6. Build and save the FAISS index

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

store = FAISS.from_documents(
    docs, embeddings, distance_strategy=DistanceStrategy.COSINE
)

store.save_local("/content/faiss_index")
print(f"Indexed {store.index.ntotal} chunks -> /content/faiss_index")

## 7. Search the index

This is the exact retrieval the local app performs. Note the relevance floor: chunks scoring far
below the best hit are dropped, so an unrelated question returns nothing instead of returning
whatever was least-bad.

In [ ]:
import warnings
TOP_K, SCORE_THRESHOLD, MIN_SCORE = 4, 0.25, 0.15

def retrieve(query, top_k=TOP_K, threshold=SCORE_THRESHOLD, min_score=MIN_SCORE):
    """Same two-gate logic as the local app.

    1. Absolute floor - reject the question entirely if the best match is weaker
       than min_score. Without this a noise-only match (e.g. 0.03) would slip
       through and put irrelevant text in the prompt.
    2. Relative floor - drop weak neighbours of a good hit.
    """
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Relevance scores must be")
        hits = store.similarity_search_with_relevance_scores(query, k=top_k)
    if not hits:
        return []
    best = max(s for _, s in hits)
    if best < min_score:
        return []
    floor = max(min_score, best * threshold)
    return [(d, s) for d, s in hits if s >= floor]

for q in ["What is the attendance requirement?",
          "What documents are required for an internship?",
          "When do the internal assessments take place?",
          "What is the capital of Japan?"]:
    hits = retrieve(q)
    print(f"\n{'='*70}\nQ: {q}\n   -> {len(hits)} chunk(s)")
    if not hits:
        print("   [No relevant chunk - the assistant will refuse to answer]")
    for doc, score in hits:
        print(f"   {score:.3f}  {doc.metadata['source']} p.{doc.metadata['page']}")
        print(f"         {doc.page_content[:100].replace(chr(10),' ')}...")

## 8. Inspect a retrieved chunk

This is the context block the local app passes to Ollama, including the source tag that makes
the final answer citable.

In [ ]:
def format_context(hits):
    blocks = []
    for i, (doc, _) in enumerate(hits, start=1):
        src = f"{doc.metadata['source']}, page {doc.metadata['page']}"
        blocks.append(f"[Chunk {i}] (Source: {src})\n{doc.page_content.strip()}")
    return "\n\n---\n\n".join(blocks)

hits = retrieve("What is the attendance requirement?")
print(format_context(hits)[:1500])

## 9. Test sample college questions

Retrieval quality per question. The **Decision** column is what the local app does: with no
relevant chunk it refuses rather than calling the LLM.

In [ ]:
TEST_QUESTIONS = [
    # In-scope: answers are present in the documents.
    "What is the attendance requirement?",
    "What are the credits for Database Management Systems?",
    "What documents are required for an internship?",
    "How is an internship evaluated?",
    "How do I register for an examination?",
    "When do the internal assessments take place?",
    # Out-of-scope: nothing in the documents answers these.
    "What is the capital of Japan?",
    "What is the hostel mess fee?",
    "Who is the hostel warden?",
    "What is the library fine amount?",
]

rows = []
for q in TEST_QUESTIONS:
    hits = retrieve(q)
    top = hits[0][1] if hits else 0.0
    rows.append({
        "Question": q,
        "Chunks": len(hits),
        "Best score": round(top, 3),
        "Top source": f"{hits[0][0].metadata['source']} p.{hits[0][0].metadata['page']}" if hits else "-",
        "Decision": "Answer from docs" if hits else "Refuse (not in docs)",
    })

df = pd.DataFrame(rows)
display(df.style.background_gradient(subset=["Best score"], cmap="RdYlGn"))
df.to_csv("/content/retrieval_results.csv", index=False)

### Score distribution

A healthy index shows a clear gap: college questions score well above the floor, unrelated
questions fall near zero. If everything scores similar, the embedding model is not separating
topics and chunk sizes need tuning.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
colors = ["#2e7d32" if v else "#c62828" for v in df["Chunks"] > 0]
ax.barh(range(len(df)), df["Best score"], color=colors)
ax.set_yticks(range(len(df)))
ax.set_yticklabels([q[:48] for q in df["Question"]], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("Best relevance score")
ax.set_title("Retrieval score per question (green = answered, red = refused)")
plt.tight_layout()
plt.show()

## 10. Tuning experiments

Retrieval quality depends on `CHUNK_SIZE`, `TOP_K` and `SCORE_THRESHOLD`. Measure, do not guess.

In [ ]:
def evaluate(chunk_size=400, chunk_overlap=80, top_k=4, threshold=0.25):
    sp = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap,
                                       separators=["\n\n", "\n", ". ", " ", ""])
    d = [Document(page_content=c.strip(), metadata=p)
         for p in pages for c in sp.split_text(p["page_content"]) if len(c.strip()) >= 20]
    st = FAISS.from_documents(d, embeddings, distance_strategy=DistanceStrategy.COSINE)
    results = []
    for q in TEST_QUESTIONS:
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="Relevance scores must be")
            hits = st.similarity_search_with_relevance_scores(q, k=top_k)
        best = max((s for _, s in hits), default=0.0)
        kept = [h for h in hits if best > 0 and h[1] >= best * threshold]
        results.append((len(d), len(kept), round(best, 3)))
    return results

for cs, co in [(400, 80), (800, 120), (1200, 200)]:
    r = evaluate(chunk_size=cs, chunk_overlap=co)
    chunks = r[0][0]
    avg_kept = sum(x[1] for x in r) / len(r)
    print(f"chunk_size={cs:5d} overlap={co:4d} -> {chunks:4d} chunks, "
          f"avg {avg_kept:.1f} kept/question")

## 11. Download the index for local use

The FAISS index built here is the same format the local app loads. Downloading it is optional —
rebuilding locally with `python -m src.rag.vector_store --build` is usually faster, since the
local build also embeds your local PDFs.

In [ ]:
import shutil
shutil.make_archive("/content/faiss_index", "zip", "/content/faiss_index")
from IPython.display import FileLink
print(FileLink("/content/faiss_index.zip"))

## Next steps on the local machine

This notebook validated retrieval. The remaining components need Ollama, which runs locally:

```bash
ollama serve
ollama pull qwen2.5:1.5b
python -m src.rag.vector_store --build
streamlit run app.py
```

Then confirm generation end to end:

```bash
python -m pytest tests/test_end_to_end.py -v
```